# ATML PA4 Task 1 full pipeline

Baseline, prune, quantize, resources, analyze on Qwen2.5-1.5B.

In [ ]:
import gzip, os, shutil, subprocess, sys
from pathlib import Path

root = Path('/kaggle/input')
print('inputs', [p.name for p in root.iterdir()], flush=True)
locked = next(root.rglob('c4_calibration_locked.jsonl*'))
source = locked.parents[2]
data = Path('/kaggle/working/pa4-data')
if data.exists():
    shutil.rmtree(data)
gzip_dirs = {'public_locked', 'kv_retrieval', 'mqar'}
for path in source.rglob('*'):
    if not path.is_file():
        continue
    rel = path.relative_to(source)
    if path.suffix == '.jsonl' and path.parent.name in gzip_dirs:
        target = data / rel.with_suffix('.jsonl.gz')
        target.parent.mkdir(parents=True, exist_ok=True)
        with path.open('rb') as raw, gzip.open(target, 'wb') as packed:
            shutil.copyfileobj(raw, packed)
    else:
        target = data / rel
        target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(path, target)
print('data', data, flush=True)

repo = Path('/kaggle/working/ATML-PA4')
if repo.exists():
    shutil.rmtree(repo)
subprocess.check_call(['git', 'clone', 'https://github.com/ttqureshi/ATML-PA4.git', str(repo)])
subprocess.check_call(['git', '-C', str(repo), 'checkout', '--detach', 'origin/task1'])
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'transformers==4.57.1', 'huggingface_hub==0.36.2', 'accelerate', 'matplotlib', 'sentencepiece', 'protobuf'])

env = os.environ.copy()
env['PA4_DATA_REPO'] = str(data)
env['PYTHONUNBUFFERED'] = '1'
env['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'
env['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

out = 'task1/results'
for stage in ['baseline', 'prune', 'quantize', 'resources', 'analyze']:
    print('\n====', stage, '====', flush=True)
    cmd = [sys.executable, 'task1/run.py', stage, '--output', out]
    rc = subprocess.run(cmd, cwd=repo, env=env).returncode
    print('exit', rc, flush=True)
    if rc != 0:
        raise SystemExit(rc)

bundle = Path('/kaggle/working/task1_results')
if bundle.exists():
    shutil.rmtree(bundle)
shutil.copytree(repo / 'task1' / 'results', bundle)
print('TASK1 COMPLETE', flush=True)
